# 15 Skills、Web Search与Multi-Agent后续路线

**用途：** 明确尚未实现的能力、添加条件和生产优先级，避免为了名词过度设计。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


In [2]:
status_rows = [
    {"能力": "受控web_search_tool", "当前": "未实现", "何时增加": "内部知识缺失且需要官方最新信息"},
    {"能力": "Skill封装", "当前": "未实现", "何时增加": "业务流程稳定、可复用且有清晰输入输出"},
    {"能力": "技术诊断sub-agent", "当前": "未实现", "何时增加": "主图复杂度和评测证明需要角色隔离"},
    {"能力": "完整multi-agent", "当前": "后置", "何时增加": "多角色并行收益大于协调成本"},
    {"能力": "FastAPI", "当前": "未实现", "何时增加": "多客户端和真实外部API"},
    {"能力": "LangSmith", "当前": "未正式接入", "何时增加": "需要统一Trace、数据集和线上评测"},
    {"能力": "Postgres checkpointer", "当前": "未实现", "何时增加": "多实例和持久部署"},
]
show_table(status_rows)
check_equal("规划项数量", len(status_rows), 7)

web_search_exists = (PROJECT2_ROOT / "tools" / "web_search_tool.py").exists()
check("没有把网页搜索误报为已实现", not web_search_exists)

,能力,当前,何时增加
0,受控web_search_tool,未实现,内部知识缺失且需要官方最新信息
1,Skill封装,未实现,业务流程稳定、可复用且有清晰输入输出
2,技术诊断sub-agent,未实现,主图复杂度和评测证明需要角色隔离
3,完整multi-agent,后置,多角色并行收益大于协调成本
4,FastAPI,未实现,多客户端和真实外部API
5,LangSmith,未正式接入,需要统一Trace、数据集和线上评测
6,Postgres checkpointer,未实现,多实例和持久部署


[PASS] 规划项数量 | actual=7, expected=7
[PASS] 没有把网页搜索误报为已实现


{'检查项': '没有把网页搜索误报为已实现', '状态': 'PASS', '说明': ''}

## Web Search设计底线

只读、官方域名白名单、保存URL和检索时间、禁止携带客户隐私、结果作为不可信证据、检测Prompt Injection、不能直接触发价格和售后结论。适合查最新官方公告、公开件号说明和物流政策，不适合搜索客户订单或把论坛内容当企业政策。

## Skills、sub-agent和multi-agent

Skill适合把稳定流程封装为可复用能力。sub-agent适合把技术诊断等高复杂度任务隔离。multi-agent只有在并行角色、独立工具权限或上下文隔离有明确收益时才值得引入；否则会增加路由、通信、Token、死循环和可观测成本。

## 部署和生产还要考虑

- Postgres checkpoint和会话目录、租户鉴权、数据保留与删除。
- 分布式限流、熔断、Provider降级、真实Token/费用回传。
- Prompt/模型/数据集版本冻结、灰度发布和回滚。
- PII脱敏、权限审计、内容安全和人工SLA。
- 离线评测、线上采样评测、badcase闭环和告警。

### 面试官会问

1. 什么情况下单Agent加工具比multi-agent更好？
2. sub-agent共享哪些State，如何限制权限？
3. Skill与普通函数、Tool有什么区别？
4. 网页搜索如何防注入和错误来源？
5. 部署后checkpoint、记忆、日志分别怎么扩展？

### 参考答案

1. **什么时候单Agent加工具更好？** 任务共享同一业务State、步骤主要串行、工具权限相近且一个显式图就能稳定路由时，单Agent更省Token、更容易测试和追踪。只有角色可并行、上下文需要隔离或权限明显不同，多Agent才可能带来净收益。
2. **sub-agent共享什么State，如何限权？** 只传任务所需的最小子集，例如已确认机型、脱敏问题和RAG证据，不共享API Key、完整客户历史或可写工具。主图通过输入/输出Schema、工具白名单、超时/预算和结果校验限制它。
3. **Skill、函数和Tool有什么区别？** 函数是代码实现；Tool是给Agent调用的带名称、描述和Schema的能力；Skill是更高层的可复用流程知识，可能包含多步操作、工具组合、规则和验收方式。不是每个函数都值得包装成Skill。
4. **网页搜索怎样防注入和错误来源？** 只允许官方域名白名单和只读GET，查询前删除客户隐私，保存URL、时间和摘要；网页内容作为不可信证据进行注入检测，事实需多源或内部规则校验，不能直接触发报价、售后或写操作。
5. **部署后如何扩展三类存储？** checkpoint和会话目录迁到带`tenant_id`的Postgres；长期记忆使用独立受治理表、过期和删除审计；日志进入集中式日志/Trace系统并做采样、脱敏、访问控制和保留策略，图片则进入对象存储。

**代码/方案落点：** 当前尚未实现这些扩展；设计记录在`docs/multimodal_web_search_roadmap.md`和`docs/web_session_architecture.md`。面试时必须明确说“已设计，未上线”。

**成功标准：** 能准确区分“已实现、已设计、后置”，并说明每项能力的触发条件。